# Hidden-Perturber Discovery (Milestone 1)

Recover an invisible body's mass and orbit from noisy trajectories of the visible bodies, via differentiable N-body simulation.

**BUILT from `src/perturber` at commit `unknown` — do not edit; edit the repo and rerun `scripts/build_notebook.py`.**

Set `RUN_SWEEP = True` in the driver cell for the full detection-threshold study (~hours on GPU).

### `src/perturber/config.py`

In [ ]:
"""Experiment configuration and presets.

Units: G = 1, star mass = 1, planet-1 semi-major axis = 1.
Planet-1 period is therefore 2*pi time units.
"""
from dataclasses import dataclass, asdict, replace  # noqa: F401  (replace used by callers)
import json

import numpy as np
import torch


@dataclass
class SystemConfig:
    # Visible bodies: index 0 is the star, then planets.
    masses_visible: tuple = (1.0, 1e-4, 3e-5)
    planet_a: tuple = (1.0, 1.9)
    planet_e: tuple = (0.0, 0.05)
    planet_phase: tuple = (0.0, 2.1)   # true anomaly at t0 [rad]
    planet_omega: tuple = (0.0, 0.8)   # argument of periapsis [rad]
    # Hidden perturber (ground truth; never shown to the model)
    hidden_mass: float = 1e-3
    hidden_a: float = 3.2
    hidden_e: float = 0.1
    hidden_phase: float = 1.0
    hidden_omega: float = 0.5
    # Observations
    n_periods: float = 16.0            # arc length in planet-1 periods
    obs_per_period: int = 40
    sigma: float = 1e-4                # position noise, absolute units
    train_frac: float = 0.75
    seed: int = 0
    # M2 only: inject a non-Newtonian central force to discover, e.g.
    # {"kind": "power_law", "params": {"alpha": 2e-3, "n": 4.0}}. None => pure M1.
    m2_force: dict = None

    @property
    def n_visible(self):
        return len(self.masses_visible)


@dataclass
class FitConfig:
    n_restarts: int = 16
    # (train-arc fraction, adam steps) single-shooting curriculum phases
    curriculum: tuple = ((0.25, 200), (0.5, 200), (1.0, 400))
    # Multiple shooting is OFF by default: on the near-Keplerian M1 systems the
    # single-shooting curriculum converges cleanly (recovers log-mass to ~0.01),
    # and the current MS tuning destabilizes an already-good fit. Retained as an
    # opt-in (set ms_steps>0) for the longer/chaotic arcs of later milestones,
    # where it will need retuning. See CLAUDE.md "Known risks".
    ms_steps: int = 0                  # multiple-shooting steps on full train arc (0 = skip)
    n_segments: int = 8                # multiple-shooting segment count
    lambda_cont: float = 1e2           # continuity penalty weight (annealed up 10x mid-phase)
    lr_elements: float = 1e-2
    lr_mass: float = 3e-2
    clip: float = 1.0
    substeps: int = 4                  # RK4 steps between consecutive observations
    softening: float = 1e-2            # distance clamp in the fitted dynamics only
    delta_prior: float = 1e-3          # weight on visible initial-state delta penalty
    mass_init: float = 1e-3
    log_a_range: tuple = (0.4, 1.8)    # restart prior: ln(a) uniform -> a in [1.5, 6.0]
    seed: int = 0
    device: str = "auto"               # "auto" | "cpu" | "cuda"


def resolve_device(spec):
    if spec == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(spec)


def get_preset(name):
    """Return (SystemConfig, FitConfig) for a named preset."""
    if name == "smoke":
        # Easy, fast case: big perturber, short arc, few restarts. ~1-2 min on CPU.
        sys_cfg = SystemConfig(n_periods=4.0, hidden_mass=1e-2, sigma=1e-4)
        fit_cfg = FitConfig(n_restarts=4, curriculum=((0.5, 80), (1.0, 120)),
                            ms_steps=0, n_segments=1, device="cpu")
    elif name == "local":
        sys_cfg = SystemConfig()
        fit_cfg = FitConfig()
    elif name == "kaggle":
        sys_cfg = SystemConfig()
        fit_cfg = FitConfig(device="auto")
    else:
        raise ValueError(f"unknown preset '{name}'")
    return sys_cfg, fit_cfg


def dump_configs(sys_cfg, fit_cfg, path):
    with open(path, "w") as f:
        json.dump({"system": asdict(sys_cfg), "fit": asdict(fit_cfg)}, f, indent=2)


def planet1_period(sys_cfg):
    mu = sys_cfg.masses_visible[0] + sys_cfg.masses_visible[1]
    return 2 * np.pi * np.sqrt(sys_cfg.planet_a[0] ** 3 / mu)


### `src/perturber/dynamics.py`

In [ ]:
"""N-body gravitational dynamics: numpy (ground truth) and torch (differentiable).

State layout: (..., N, 4) with [:2] = position (x, y) and [2:] = velocity (vx, vy).
G = 1 everywhere.
"""
import numpy as np
import torch

G = 1.0


# ── numpy side (ground-truth integration via scipy) ──────────────────────────

def accel_numpy(pos, masses, extra_force=None):
    """pos (N, 2), masses (N,) -> acceleration (N, 2). Pure gravity, no softening.

    extra_force(pos) -> (N, 2) adds a non-Newtonian term (M2 injected law)."""
    diff = pos[None, :, :] - pos[:, None, :]          # diff[i, j] = r_j - r_i
    d2 = (diff ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    inv_d3 = d2 ** -1.5
    acc = G * (diff * (masses[None, :, None] * inv_d3[:, :, None])).sum(axis=1)
    if extra_force is not None:
        acc = acc + extra_force(pos)
    return acc


def ode_rhs(t, y, masses, extra_force=None):
    """Flat RHS for scipy.integrate.solve_ivp. y = [pos.ravel(), vel.ravel()]."""
    n = len(masses)
    pos = y[: 2 * n].reshape(n, 2)
    vel = y[2 * n:].reshape(n, 2)
    return np.concatenate([vel.ravel(),
                           accel_numpy(pos, masses, extra_force=extra_force).ravel()])


# ── torch side (differentiable, batched) ──────────────────────────────────────

def accel_torch(pos, masses, softening=0.0, extra_force=None):
    """pos (..., N, 2), masses (..., N) or (N,) -> acceleration (..., N, 2).

    softening clamps pairwise distance^2 (fitted dynamics only; truth uses 0).
    extra_force(pos, vel=None) is the milestone-2 hook for a neural residual force.
    """
    diff = pos.unsqueeze(-3) - pos.unsqueeze(-2)       # (..., N, N, 2), diff[i, j] = r_j - r_i
    d2 = (diff ** 2).sum(-1)                           # (..., N, N)
    n = pos.shape[-2]
    eye = torch.eye(n, dtype=pos.dtype, device=pos.device)
    d2 = d2.clamp(min=softening ** 2) + eye            # +eye avoids 0^-1.5 on the diagonal
    inv_d3 = d2.pow(-1.5) * (1.0 - eye)                # zero self-interaction
    m_j = masses.unsqueeze(-2)                         # (..., 1, N)
    acc = G * (diff * (m_j * inv_d3).unsqueeze(-1)).sum(-2)
    if extra_force is not None:
        acc = acc + extra_force(pos)
    return acc


def rhs_torch(state, masses, softening=0.0, extra_force=None):
    """state (..., N, 4) -> d(state)/dt (..., N, 4)."""
    pos, vel = state[..., :2], state[..., 2:]
    acc = accel_torch(pos, masses, softening=softening, extra_force=extra_force)
    return torch.cat([vel, acc], dim=-1)


def total_energy(state, masses):
    """Diagnostic: kinetic + potential energy. state (N, 4) numpy, masses (N,)."""
    pos, vel = state[:, :2], state[:, 2:]
    ke = 0.5 * (masses * (vel ** 2).sum(-1)).sum()
    diff = pos[None, :, :] - pos[:, None, :]
    d = np.sqrt((diff ** 2).sum(-1))
    np.fill_diagonal(d, np.inf)
    pe = -0.5 * G * (masses[:, None] * masses[None, :] / d).sum()
    return ke + pe


### `src/perturber/forces.py`

In [ ]:
"""Milestone 2 — non-Newtonian force laws to inject and discover.

A central extra force anchored on the dominant mass (the star, body 0):

    a_extra(body i) = -g(r_i) * rhat_i ,   r_i = |pos_i - pos_star|

with rhat_i the unit vector from star to body i (so positive g is *attractive*,
an extra inward pull). The discovery target in M2 is the radial profile g(r).

Injected truth uses a power law g(r) = alpha / r^n (n != 2 is genuinely
non-Newtonian; n = 2 would be degenerate with the star's mass). The star itself
feels no extra force here (a perturbative external-field approximation — valid
because the star dominates the mass and the term is small; momentum is not
exactly conserved, which is acceptable for a synthetic discovery target).
"""
import numpy as np
import torch


def _radial_apply_np(pos, star_idx, g_of_r):
    """pos (N,2) -> extra accel (N,2). g_of_r: array r -> array g."""
    d = pos - pos[star_idx][None, :]          # (N,2) from star
    r = np.linalg.norm(d, axis=1)             # (N,)
    out = np.zeros_like(pos)
    mask = np.arange(len(pos)) != star_idx
    rr = r[mask]
    rhat = d[mask] / rr[:, None]
    out[mask] = -(g_of_r(rr)[:, None]) * rhat
    return out


def _radial_apply_torch(pos, star_idx, g_of_r):
    """pos (...,N,2) -> extra accel (...,N,2). g_of_r: tensor r -> tensor g."""
    d = pos - pos[..., star_idx:star_idx + 1, :]
    r = torch.linalg.norm(d, dim=-1).clamp(min=1e-9)      # (...,N)
    rhat = d / r.unsqueeze(-1)
    g = g_of_r(r)                                         # (...,N)
    out = -g.unsqueeze(-1) * rhat
    # zero the star's own entry
    n = pos.shape[-2]
    keep = torch.ones(n, dtype=pos.dtype, device=pos.device)
    keep[star_idx] = 0.0
    return out * keep.unsqueeze(-1)


def power_law(alpha=1e-3, n=4.0, star_idx=0, backend="numpy"):
    """Injected truth: g(r) = alpha / r^n. Returns an extra_force(pos) callable."""
    if backend == "numpy":
        return lambda pos: _radial_apply_np(pos, star_idx, lambda r: alpha / r ** n)
    return lambda pos: _radial_apply_torch(pos, star_idx, lambda r: alpha / r ** n)


def yukawa(alpha=1e-3, lam=1.0, star_idx=0, backend="numpy"):
    """Injected truth: g(r) = alpha * exp(-r/lam) / r^2 (screened gravity)."""
    if backend == "numpy":
        return lambda pos: _radial_apply_np(
            pos, star_idx, lambda r: alpha * np.exp(-r / lam) / r ** 2)
    return lambda pos: _radial_apply_torch(
        pos, star_idx, lambda r: alpha * torch.exp(-r / lam) / r ** 2)


def make_injected(kind, params, star_idx=0, backend="numpy"):
    if kind == "power_law":
        return power_law(star_idx=star_idx, backend=backend, **params)
    if kind == "yukawa":
        return yukawa(star_idx=star_idx, backend=backend, **params)
    raise ValueError(kind)


### `src/perturber/data.py`

In [ ]:
"""Synthetic dataset generation: ground truth, noisy observations, splits."""
from dataclasses import dataclass

import numpy as np
from scipy.integrate import solve_ivp



def kepler_state(a, e, nu, omega, mu):
    """2-D orbital elements -> (pos(2,), vel(2,)) relative to the central body."""
    p = a * (1.0 - e * e)
    r = p / (1.0 + e * np.cos(nu))
    pos_o = np.array([r * np.cos(nu), r * np.sin(nu)])
    vf = np.sqrt(mu / p)
    vel_o = vf * np.array([-np.sin(nu), e + np.cos(nu)])
    c, s = np.cos(omega), np.sin(omega)
    rot = np.array([[c, -s], [s, c]])
    return rot @ pos_o, rot @ vel_o


def initial_state(cfg: SystemConfig):
    """Full-system initial state (N, 4) in the centre-of-momentum frame.

    N = n_visible + 1; the hidden perturber is the last body.
    """
    m_star = cfg.masses_visible[0]
    bodies = [np.zeros(4)]  # star at origin, at rest (pre-COM correction)
    for i in range(1, cfg.n_visible):
        pos, vel = kepler_state(cfg.planet_a[i - 1], cfg.planet_e[i - 1],
                                cfg.planet_phase[i - 1], cfg.planet_omega[i - 1],
                                mu=m_star + cfg.masses_visible[i])
        bodies.append(np.concatenate([pos, vel]))
    pos, vel = kepler_state(cfg.hidden_a, cfg.hidden_e, cfg.hidden_phase,
                            cfg.hidden_omega, mu=m_star + cfg.hidden_mass)
    bodies.append(np.concatenate([pos, vel]))
    state = np.stack(bodies)                                  # (N, 4)

    masses = np.array(list(cfg.masses_visible) + [cfg.hidden_mass])
    com = (masses[:, None] * state).sum(0) / masses.sum()     # COM pos and momentum/M
    return state - com[None, :], masses


@dataclass
class Dataset:
    t: np.ndarray            # (T,) observation times
    truth: np.ndarray        # (T, N, 4) all bodies, noiseless (hidden body last)
    q_obs: np.ndarray        # (T, Nv, 2) noisy visible positions
    train_mask: np.ndarray   # (T,) bool — first train_frac of the arc
    test_mask: np.ndarray    # (T,) bool — held-out future window
    sys: SystemConfig

    @property
    def n_visible(self):
        return self.sys.n_visible


def generate(cfg: SystemConfig) -> Dataset:
    rng = np.random.default_rng(cfg.seed)
    state0, masses = initial_state(cfg)

    p1 = planet1_period(cfg)
    t_end = cfg.n_periods * p1
    n_obs = int(round(cfg.n_periods * cfg.obs_per_period)) + 1
    t = np.linspace(0.0, t_end, n_obs)

    extra = None
    if cfg.m2_force is not None:
        extra = make_injected(cfg.m2_force["kind"], cfg.m2_force["params"],
                              backend="numpy")

    sol = solve_ivp(ode_rhs, (0.0, t_end), _flat(state0),
                    t_eval=t, args=(masses, extra), method="DOP853",
                    rtol=1e-11, atol=1e-12)
    assert sol.success, sol.message
    n = len(masses)
    pos_t = np.moveaxis(sol.y[: 2 * n].reshape(n, 2, -1), -1, 0)   # (T, N, 2)
    vel_t = np.moveaxis(sol.y[2 * n:].reshape(n, 2, -1), -1, 0)
    truth = np.concatenate([pos_t, vel_t], axis=-1)                # (T, N, 4)

    nv = cfg.n_visible
    q_obs = truth[:, :nv, :2] + cfg.sigma * rng.standard_normal((len(t), nv, 2))

    n_train = int(round(cfg.train_frac * len(t)))
    train_mask = np.zeros(len(t), dtype=bool)
    train_mask[:n_train] = True

    # Energy-conservation sanity on the ground truth (pure-gravity only; the M2
    # injected force adds an unaccounted potential, so skip the check there).
    if cfg.m2_force is None:
        e0 = total_energy(truth[0], masses)
        e1 = total_energy(truth[-1], masses)
        assert abs(e1 - e0) / abs(e0) < 1e-8, "ground-truth integration drifted"

    return Dataset(t=t, truth=truth, q_obs=q_obs,
                   train_mask=train_mask, test_mask=~train_mask, sys=cfg)


def _flat(state):
    """(N, 4) -> flat [pos.ravel(), vel.ravel()] as ode_rhs expects."""
    return np.concatenate([state[:, :2].ravel(), state[:, 2:].ravel()])


def estimate_visible_state0(ds: Dataset, n_fit=11, deg=4):
    """Estimate visible bodies' initial positions/velocities from the first
    few noisy observations (quartic fit per coordinate — the higher degree
    keeps orbital-curvature truncation bias below the noise). Returns (Nv, 4)."""
    t = ds.t[:n_fit] - ds.t[0]
    out = np.zeros((ds.n_visible, 4))
    for b in range(ds.n_visible):
        for c in range(2):
            coef = np.polyfit(t, ds.q_obs[:n_fit, b, c], deg=deg)
            out[b, c] = coef[-1]           # value at t0
            out[b, 2 + c] = coef[-2]       # slope at t0
    return out


### `src/perturber/integrators.py`

In [ ]:
"""Batched differentiable fixed-step RK4 integrator (pure torch)."""
import torch
from torch.utils.checkpoint import checkpoint



def _interval(s, h, masses, substeps, softening, extra_force):
    """Advance one observation interval with `substeps` RK4 steps."""
    for _ in range(substeps):
        k1 = rhs_torch(s, masses, softening=softening, extra_force=extra_force)
        k2 = rhs_torch(s + 0.5 * h * k1, masses, softening=softening, extra_force=extra_force)
        k3 = rhs_torch(s + 0.5 * h * k2, masses, softening=softening, extra_force=extra_force)
        k4 = rhs_torch(s + h * k3, masses, softening=softening, extra_force=extra_force)
        s = s + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return s


def rk4_integrate(state0, masses, t_grid, substeps=4, softening=0.0,
                  extra_force=None, checkpointing=False):
    """Integrate N-body dynamics on a fixed time grid.

    state0   (..., N, 4)   initial state (arbitrary batch dims)
    masses   (..., N)      broadcastable against state0's batch dims
    t_grid   (T,)          observation times (need not be uniform)
    substeps               RK4 steps between consecutive grid points

    Returns trajectory (T, ..., N, 4) — time axis first, gradients flow to
    state0 and masses through the unrolled loop.

    With `checkpointing`, the inner sub-steps of each interval are recomputed
    during backward instead of stored, bounding peak memory to ~one interval's
    graph regardless of arc length (essential for long arcs / many parallel
    fits). Costs one extra forward pass (~1.5x wall). Skipped automatically when
    the graph isn't needed (no grad).
    """
    out = [state0]
    s = state0
    use_ckpt = checkpointing and torch.is_grad_enabled() and state0.requires_grad
    for i in range(len(t_grid) - 1):
        h = (t_grid[i + 1] - t_grid[i]) / substeps
        if use_ckpt:
            s = checkpoint(_interval, s, h, masses, substeps, softening,
                           extra_force, use_reentrant=False)
        else:
            s = _interval(s, h, masses, substeps, softening, extra_force)
        out.append(s)
    return torch.stack(out, dim=0)


### `src/perturber/model.py`

In [ ]:
"""Learnable models: hidden perturber (mass + orbital elements) and the null model.

Both expose the same interface, batched over B restart candidates:
    masses()        -> (B, N)
    initial_state() -> (B, N, 4)
    delta_penalty() -> (B,)   weak prior pulling visible-state deltas to zero
"""
import math

import torch
import torch.nn as nn


class NullModel(nn.Module):
    """No hidden body — only small corrections to the observed initial states.

    This is the baseline any detection claim must beat on held-out data.
    """

    def __init__(self, n_restarts, vis_state0, masses_visible, sigma, dt_obs):
        super().__init__()
        b = n_restarts
        self.register_buffer("base", vis_state0.unsqueeze(0).repeat(b, 1, 1))  # (B, Nv, 4)
        self.register_buffer("m_vis", masses_visible.unsqueeze(0).repeat(b, 1))
        # Raw deltas are unitless; scales convert to physical units so one lr fits all.
        scale = torch.tensor([5 * sigma, 5 * sigma,
                              10 * sigma / dt_obs, 10 * sigma / dt_obs])
        self.register_buffer("delta_scale", scale)
        self.vis_delta = nn.Parameter(torch.zeros(b, vis_state0.shape[0], 4))

    def visible_state0(self):
        return self.base + self.vis_delta * self.delta_scale

    def masses(self):
        return self.m_vis

    def initial_state(self):
        return self.visible_state0()

    def delta_penalty(self):
        return (self.vis_delta ** 2).mean(dim=(1, 2))


class PerturberModel(NullModel):
    """Null model + one hidden body: learnable log10-mass and 2-D orbital
    elements (log-a, bounded e, true anomaly, argument of periapsis)."""

    def __init__(self, n_restarts, vis_state0, masses_visible, sigma, dt_obs,
                 mass_init=1e-3, log_a_range=(0.4, 1.8), generator=None):
        super().__init__(n_restarts, vis_state0, masses_visible, sigma, dt_obs)
        b = n_restarts
        g = generator
        lo, hi = log_a_range
        self.log10_mass = nn.Parameter(
            torch.full((b,), math.log10(mass_init)))
        self.log_a = nn.Parameter(lo + (hi - lo) * torch.rand(b, generator=g))
        self.raw_e = nn.Parameter(torch.full((b,), -2.0))     # sigmoid/2 -> e ~ 0.06
        self.phase = nn.Parameter(2 * math.pi * torch.rand(b, generator=g))
        self.omega = nn.Parameter(2 * math.pi * torch.rand(b, generator=g))

    def hidden_mass(self):
        return 10.0 ** self.log10_mass                        # (B,)

    def hidden_elements(self):
        return dict(a=torch.exp(self.log_a),
                    e=0.5 * torch.sigmoid(self.raw_e),
                    nu=self.phase, omega=self.omega)

    def masses(self):
        return torch.cat([self.m_vis, self.hidden_mass().unsqueeze(-1)], dim=-1)

    def initial_state(self):
        vis = self.visible_state0()                           # (B, Nv, 4)
        el = self.hidden_elements()
        a, e, nu, om = el["a"], el["e"], el["nu"], el["omega"]
        mu = self.m_vis[:, 0] + self.hidden_mass()

        p = a * (1.0 - e * e)
        r = p / (1.0 + e * torch.cos(nu))
        pos_o = torch.stack([r * torch.cos(nu), r * torch.sin(nu)], dim=-1)
        vf = torch.sqrt(mu / p)
        vel_o = vf.unsqueeze(-1) * torch.stack([-torch.sin(nu), e + torch.cos(nu)], dim=-1)
        c, s = torch.cos(om), torch.sin(om)
        rot = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], dim=-2)
        pos = (rot @ pos_o.unsqueeze(-1)).squeeze(-1) + vis[:, 0, :2]   # orbit the star
        vel = (rot @ vel_o.unsqueeze(-1)).squeeze(-1) + vis[:, 0, 2:]

        hidden = torch.cat([pos, vel], dim=-1).unsqueeze(1)   # (B, 1, 4)
        return torch.cat([vis, hidden], dim=1)


### `src/perturber/fit.py`

In [ ]:
"""Fitting engine: batched multi-start, short-arc curriculum, multiple shooting.

Everything runs in float64 — observation noise is 1e-4 in O(1) coordinates and
float32 rounding accumulated over thousands of RK4 steps would eat the signal.
"""
import time
from dataclasses import dataclass

import numpy as np
import torch



def make_model(kind, ds: Dataset, cfg: FitConfig):
    device = resolve_device(cfg.device)
    vis0 = torch.tensor(estimate_visible_state0(ds), dtype=torch.float64)
    m_vis = torch.tensor(np.array(ds.sys.masses_visible), dtype=torch.float64)
    dt_obs = float(ds.t[1] - ds.t[0])
    gen = torch.Generator().manual_seed(cfg.seed)
    if kind == "perturber":
        model = PerturberModel(cfg.n_restarts, vis0, m_vis, ds.sys.sigma, dt_obs,
                               mass_init=cfg.mass_init, log_a_range=cfg.log_a_range,
                               generator=gen)
    elif kind == "null":
        model = NullModel(cfg.n_restarts, vis0, m_vis, ds.sys.sigma, dt_obs)
    else:
        raise ValueError(kind)
    return model.double().to(device)


def chi2_per_candidate(traj, q_obs, sigma):
    """traj (T, B, N, 4), q_obs (T, Nv, 2) -> mean normalized sq residual (B,)."""
    nv = q_obs.shape[1]
    res = (traj[:, :, :nv, :2] - q_obs.unsqueeze(1)) / sigma
    return (res ** 2).mean(dim=(0, 2, 3))


@dataclass
class FitResult:
    best_idx: int
    train_chi2: np.ndarray      # (B,) final single-shot train chi2 per candidate
    history: list               # (phase, step, best loss, [mass spread]) tuples
    wall_time: float


def _make_optimizer(model, cfg, extra_params=()):
    fast, slow = [], list(extra_params)
    for name, p in model.named_parameters():
        (fast if name == "log10_mass" else slow).append(p)
    groups = [{"params": slow, "lr": cfg.lr_elements}]
    if fast:
        groups.append({"params": fast, "lr": cfg.lr_mass})
    return torch.optim.Adam(groups)


def _clamp_(model, cfg):
    if isinstance(model, PerturberModel):
        with torch.no_grad():
            model.log10_mass.clamp_(-6.0, -0.5)
            lo, hi = cfg.log_a_range
            model.log_a.clamp_(lo - 0.3, hi + 0.3)


def _log_line(model, tag, step, loss_b):
    best = loss_b.min().item()
    msg = f"  [{tag}] step {step:4d} | best chi2 {best:9.3f}"
    if isinstance(model, PerturberModel):
        m = model.hidden_mass().detach().cpu().numpy()
        msg += f" | log10 m: {np.log10(m).min():.2f}..{np.log10(m).max():.2f}"
    print(msg, flush=True)


def fit(model, ds: Dataset, cfg: FitConfig, verbose=True):
    device = next(model.parameters()).device
    t_all = torch.tensor(ds.t, dtype=torch.float64, device=device)
    q_all = torch.tensor(ds.q_obs, dtype=torch.float64, device=device)
    n_train = int(ds.train_mask.sum())
    sigma = ds.sys.sigma
    history = []
    t_start = time.time()

    # ── Phase A: single-shooting curriculum on growing train arcs ────────────
    opt = _make_optimizer(model, cfg)
    for phase_i, (frac, steps) in enumerate(cfg.curriculum):
        n_sub = int(round(frac * (n_train - 1))) + 1
        t_sub, q_sub = t_all[:n_sub], q_all[:n_sub]
        for step in range(steps):
            opt.zero_grad()
            traj = rk4_integrate(model.initial_state(), model.masses(), t_sub,
                                 substeps=cfg.substeps, softening=cfg.softening)
            loss_b = chi2_per_candidate(traj, q_sub, sigma) \
                + cfg.delta_prior * model.delta_penalty()
            loss_b.sum().backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip)
            opt.step()
            _clamp_(model, cfg)
            if verbose and step % max(1, steps // 4) == 0:
                _log_line(model, f"phase {phase_i} frac {frac}", step, loss_b)
        history.append(("curriculum", frac, float(loss_b.min())))

    # ── Phase B: multiple shooting on the full train arc ─────────────────────
    if cfg.ms_steps > 0 and cfg.n_segments > 1:
        k = cfg.n_segments
        assert (n_train - 1) % k == 0, \
            f"train intervals {n_train - 1} not divisible by n_segments {k}"
        seg_len = (n_train - 1) // k
        b = model.vis_delta.shape[0]
        n_bodies = model.masses().shape[-1]

        # Initialize interior segment starts from the current best integration.
        with torch.no_grad():
            traj = rk4_integrate(model.initial_state(), model.masses(),
                                 t_all[:n_train], substeps=cfg.substeps,
                                 softening=cfg.softening)
        seg_states = torch.nn.Parameter(
            traj[[i * seg_len for i in range(1, k)]].permute(1, 0, 2, 3)
            .contiguous())                                   # (B, K-1, N, 4)

        # State-space scales for the continuity penalty
        dt_obs = float(ds.t[1] - ds.t[0])
        cont_scale = torch.tensor([sigma, sigma, sigma / dt_obs, sigma / dt_obs],
                                  dtype=torch.float64, device=device)
        t_seg = t_all[: seg_len + 1]                         # uniform grid: shared offsets
        q_segs = torch.stack([q_all[i * seg_len: i * seg_len + seg_len + 1]
                              for i in range(k)])            # (K, T_seg, Nv, 2)

        opt = _make_optimizer(model, cfg, extra_params=[seg_states])
        for step in range(cfg.ms_steps):
            opt.zero_grad()
            starts = torch.cat([model.initial_state().unsqueeze(1), seg_states],
                               dim=1)                        # (B, K, N, 4)
            flat = starts.reshape(b * k, n_bodies, 4)
            m_flat = model.masses().unsqueeze(1).expand(b, k, n_bodies) \
                                   .reshape(b * k, n_bodies)
            traj = rk4_integrate(flat, m_flat, t_seg, substeps=cfg.substeps,
                                 softening=cfg.softening)    # (T_seg, B*K, N, 4)
            traj = traj.reshape(-1, b, k, n_bodies, 4)

            nv = q_all.shape[1]
            res = (traj[:, :, :, :nv, :2]
                   - q_segs.permute(1, 0, 2, 3).unsqueeze(1)) / sigma
            data_b = (res ** 2).mean(dim=(0, 2, 3, 4))       # (B,)

            gap = (traj[-1, :, :-1] - seg_states) / cont_scale
            lam = cfg.lambda_cont * (10.0 if step > cfg.ms_steps // 2 else 1.0)
            cont_b = (gap ** 2).mean(dim=(1, 2, 3))

            loss_b = data_b + lam * cont_b + cfg.delta_prior * model.delta_penalty()
            loss_b.sum().backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + [seg_states], cfg.clip)
            opt.step()
            _clamp_(model, cfg)
            if verbose and step % max(1, cfg.ms_steps // 4) == 0:
                _log_line(model, "mshoot", step, data_b)
        history.append(("mshoot", 1.0, float(data_b.min())))

    # ── Final honest selection: single-shot from t0 over the full train arc ──
    with torch.no_grad():
        traj = rk4_integrate(model.initial_state(), model.masses(),
                             t_all[:n_train], substeps=cfg.substeps,
                             softening=cfg.softening)
        train_chi2 = chi2_per_candidate(traj, q_all[:n_train], sigma) \
            .cpu().numpy()
    best_idx = int(train_chi2.argmin())
    if verbose:
        order = np.argsort(train_chi2)
        print(f"  [select] train chi2 per restart (sorted): "
              f"{np.array2string(train_chi2[order][:5], precision=2)}")
    return FitResult(best_idx=best_idx, train_chi2=train_chi2,
                     history=history, wall_time=time.time() - t_start)


### `src/perturber/evaluate.py`

In [ ]:
"""Evaluation: parameter recovery, forward prediction, null-model comparison."""
import numpy as np
import torch



def integrate_best(model, result, ds, cfg):
    """Single-shot integrate the best restart over the FULL arc (train + held-out).

    Returns trajectory (T, N, 4) numpy. No softening here — evaluation uses the
    same pure dynamics as the ground truth.
    """
    device = next(model.parameters()).device
    t_all = torch.tensor(ds.t, dtype=torch.float64, device=device)
    with torch.no_grad():
        traj = rk4_integrate(model.initial_state(), model.masses(), t_all,
                             substeps=cfg.substeps, softening=0.0)
    return traj[:, result.best_idx].cpu().numpy()


def evaluate(model, result, ds, cfg):
    """Metrics dict for one fitted model (perturber or null)."""
    traj = integrate_best(model, result, ds, cfg)
    nv = ds.n_visible
    tr, te = ds.train_mask, ds.test_mask
    sigma = ds.sys.sigma

    def chi2(mask):
        r = (traj[mask][:, :nv, :2] - ds.q_obs[mask]) / sigma
        return float((r ** 2).mean())

    def rmse_truth(mask):
        d = traj[mask][:, :nv, :2] - ds.truth[mask][:, :nv, :2]
        return float(np.sqrt((d ** 2).sum(-1).mean()))

    metrics = {
        "chi2_train": chi2(tr),
        "chi2_test": chi2(te),
        "rmse_truth_train": rmse_truth(tr),
        "rmse_truth_test": rmse_truth(te),
        "n_obs_test": int(te.sum()) * nv * 2,
        "wall_time_s": result.wall_time,
        "restart_train_chi2": result.train_chi2.tolist(),
    }

    if isinstance(model, PerturberModel):
        i = result.best_idx
        with torch.no_grad():
            m_hat = float(model.hidden_mass()[i])
            a_hat = float(model.hidden_elements()["a"][i])
        m_true, a_true = ds.sys.hidden_mass, ds.sys.hidden_a
        mid = len(ds.t) // 2
        pos_err = float(np.linalg.norm(traj[mid, -1, :2] - ds.truth[mid, -1, :2]))
        # Restart agreement: log-mass spread of the top-3 candidates
        order = np.argsort(result.train_chi2)[:3]
        with torch.no_grad():
            top_masses = model.hidden_mass().cpu().numpy()[order]
        metrics.update({
            "mass_true": m_true,
            "mass_recovered": m_hat,
            "log_mass_error": abs(np.log10(m_hat / m_true)),
            "a_true": a_true,
            "a_recovered": a_hat,
            "a_rel_error": abs(a_hat - a_true) / a_true,
            "hidden_pos_error_midarc": pos_err,
            "top3_log10_mass": np.log10(top_masses).tolist(),
        })
    return metrics, traj


def compare(metrics_pert, metrics_null):
    """Detection statistic: does the perturber model beat the null on held-out
    data? Under the Gaussian noise model, 2*delta(logL) = n_test * delta(chi2)."""
    d_chi2 = metrics_null["chi2_test"] - metrics_pert["chi2_test"]
    stat = metrics_pert["n_obs_test"] * d_chi2
    return {
        "delta_chi2_test": d_chi2,
        "delta_rmse_test": metrics_null["rmse_truth_test"] - metrics_pert["rmse_truth_test"],
        "loglike_ratio_stat": stat,       # ~2*(logL_pert - logL_null)
        "detected": bool(stat > 25.0),    # ≈5-sigma-equivalent threshold, 1 extra dof family
    }


### `src/perturber/plots.py`

In [ ]:
"""Figures: single-experiment diagnostic panel and threshold-study heatmaps."""
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def plot_experiment(ds, traj_pert, traj_null, metrics_pert, metrics_null, cmp, path):
    """Six-panel diagnostic figure for one experiment."""
    t = ds.t
    tr, te = ds.train_mask, ds.test_mask
    nv = ds.n_visible
    t_split = t[tr][-1]
    p_per = t / t[-1] * ds.sys.n_periods

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    det = "DETECTED" if cmp["detected"] else "not detected"
    fig.suptitle(
        f"Hidden-perturber discovery | m_true={ds.sys.hidden_mass:.1e} "
        f"m_hat={metrics_pert['mass_recovered']:.2e} | "
        f"sigma={ds.sys.sigma:.0e} | arc={ds.sys.n_periods:g} periods | {det}",
        fontweight="bold")

    # (0,0) orbits in XY
    ax = axes[0, 0]
    colors = ["#f39c12", "#2980b9", "#27ae60"]
    for b in range(nv):
        ax.plot(ds.truth[:, b, 0], ds.truth[:, b, 1], color=colors[b], lw=2,
                alpha=0.4, label=f"truth body {b}")
        ax.plot(traj_pert[:, b, 0], traj_pert[:, b, 1], color=colors[b], lw=1, ls="--")
    ax.plot(ds.truth[:, -1, 0], ds.truth[:, -1, 1], "k-", lw=2, alpha=0.4,
            label="hidden (truth)")
    ax.plot(traj_pert[:, -1, 0], traj_pert[:, -1, 1], "r--", lw=1.2,
            label="hidden (recovered)")
    ax.set_aspect("equal"); ax.legend(fontsize=7); ax.set_title("Orbits (XY)")

    # (0,1) planet-2 residuals vs time, both models
    ax = axes[0, 1]
    b = nv - 1
    for traj, lbl, c in [(traj_null, "null model", "gray"),
                         (traj_pert, "perturber model", "tab:blue")]:
        r = np.linalg.norm(traj[:, b, :2] - ds.truth[:, b, :2], axis=1)
        ax.semilogy(p_per, np.maximum(r, 1e-12), color=c, lw=1.2, label=lbl)
    ax.axhline(ds.sys.sigma, color="k", ls=":", lw=1, label="noise sigma")
    ax.axvline(p_per[tr][-1], color="red", ls=":", lw=1)
    ax.set_xlabel("planet-1 periods"); ax.set_title(f"Body {b} error vs truth")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # (0,2) star wobble: the mass signal
    ax = axes[0, 2]
    ax.plot(p_per, ds.q_obs[:, 0, 0], ".", ms=2, alpha=0.3, color="gray", label="obs")
    ax.plot(p_per, ds.truth[:, 0, 0], "g-", lw=1.5, alpha=0.7, label="truth")
    ax.plot(p_per, traj_pert[:, 0, 0], "b--", lw=1, label="perturber fit")
    ax.plot(p_per, traj_null[:, 0, 0], "-", color="orange", lw=1, label="null fit")
    ax.axvline(p_per[tr][-1], color="red", ls=":", lw=1)
    ax.set_xlabel("planet-1 periods"); ax.set_title("Star x (reflex wobble)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # (1,0) held-out forward prediction error, all visible bodies
    ax = axes[1, 0]
    for traj, lbl, c in [(traj_null, "null", "gray"), (traj_pert, "perturber", "tab:blue")]:
        r = np.linalg.norm(traj[:, :nv, :2] - ds.truth[:, :nv, :2], axis=2).mean(1)
        ax.semilogy(p_per, np.maximum(r, 1e-12), color=c, lw=1.2, label=lbl)
    ax.axvspan(p_per[te][0], p_per[-1], alpha=0.08, color="red", label="held-out")
    ax.axhline(ds.sys.sigma, color="k", ls=":", lw=1)
    ax.set_xlabel("planet-1 periods"); ax.set_title("Mean visible-body error")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # (1,1) restart agreement
    ax = axes[1, 1]
    chi2s = np.array(metrics_pert["restart_train_chi2"])
    masses = np.array(metrics_pert["top3_log10_mass"])
    ax.plot(np.sort(chi2s), "o-", color="tab:blue")
    ax.set_yscale("log"); ax.set_xlabel("restart (sorted)")
    ax.set_title(f"Restart train chi2 | top-3 log10(m): "
                 f"{np.array2string(masses, precision=2)}")
    ax.grid(alpha=0.3)

    # (1,2) summary text
    ax = axes[1, 2]; ax.axis("off")
    lines = [
        f"log-mass error      {metrics_pert['log_mass_error']:.3f}",
        f"a: true {metrics_pert['a_true']:.2f}  rec {metrics_pert['a_recovered']:.2f}",
        f"hidden pos err mid  {metrics_pert['hidden_pos_error_midarc']:.3f}",
        "",
        f"chi2 test  pert {metrics_pert['chi2_test']:.2f}  null {metrics_null['chi2_test']:.2f}",
        f"rmse test  pert {metrics_pert['rmse_truth_test']:.2e}  null {metrics_null['rmse_truth_test']:.2e}",
        f"2*dlogL = {cmp['loglike_ratio_stat']:.1f}  ->  {det}",
    ]
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=11)

    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight"); plt.close(fig)
    return path


def plot_threshold(cells, path):
    """Threshold-study summary. cells: list of result dicts, each with keys
    mass, sigma, n_periods, seed, log_mass_error, detected."""
    arcs = sorted({c["n_periods"] for c in cells})
    masses = sorted({c["mass"] for c in cells})
    sigmas = sorted({c["sigma"] for c in cells})

    fig, axes = plt.subplots(1, len(arcs), figsize=(5.5 * len(arcs), 4.5),
                             squeeze=False)
    for ai, arc in enumerate(arcs):
        ax = axes[0, ai]
        grid = np.full((len(masses), len(sigmas)), np.nan)
        det = np.zeros_like(grid, dtype=bool)
        for mi, m in enumerate(masses):
            for si, s in enumerate(sigmas):
                sel = [c for c in cells
                       if c["mass"] == m and c["sigma"] == s and c["n_periods"] == arc]
                if sel:
                    grid[mi, si] = np.median([c["log_mass_error"] for c in sel])
                    det[mi, si] = np.median([c["detected"] for c in sel]) >= 0.5
        im = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis_r",
                       vmin=0, vmax=2)
        for mi in range(len(masses)):
            for si in range(len(sigmas)):
                if np.isnan(grid[mi, si]):
                    continue
                mark = "" if det[mi, si] else "  X"
                ax.text(si, mi, f"{grid[mi, si]:.2f}{mark}", ha="center",
                        va="center", fontsize=9,
                        color="white" if grid[mi, si] > 1 else "black")
        ax.set_xticks(range(len(sigmas)), [f"{s:.0e}" for s in sigmas])
        ax.set_yticks(range(len(masses)), [f"{m:.0e}" for m in masses])
        ax.set_xlabel("noise sigma"); ax.set_ylabel("true hidden mass")
        ax.set_title(f"arc = {arc:g} periods  (X = not detected)")
    fig.colorbar(im, ax=axes[0, -1], label="median |log10(m_hat/m_true)|")
    fig.suptitle("Detection threshold: median log-mass error", fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight"); plt.close(fig)
    return path


def _floor(cells, arc, s, masses, predicate):
    """Smallest true mass at (arc, sigma) whose seed-median satisfies predicate."""
    ok = []
    for m in masses:
        sel = [c for c in cells if c["mass"] == m and c["sigma"] == s
               and c["n_periods"] == arc]
        if sel and predicate(sel):
            ok.append(m)
    return min(ok) if ok else None


def plot_detection_boundary(cells, path, char_thresh=0.3):
    """Two sensitivity floors vs noise, one colour per arc length:

    - solid  = detection floor: smallest mass the perturber model still beats the
      null on held-out data (median over seeds).
    - dashed = characterization floor: smallest mass recovered to within
      |log10(m_hat/m_true)| < char_thresh (~2x) — the practically useful limit.

    Lower = more sensitive. A floor at the smallest grid mass means the true
    floor is below the grid (not yet located)."""
    arcs = sorted({c["n_periods"] for c in cells})
    sigmas = sorted({c["sigma"] for c in cells})
    masses = sorted({c["mass"] for c in cells})
    colors = plt.cm.viridis(np.linspace(0.1, 0.8, len(arcs)))

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for arc, col in zip(arcs, colors):
        det_x, det_y, char_x, char_y = [], [], [], []
        for s in sigmas:
            d = _floor(cells, arc, s, masses,
                       lambda sel: np.median([c["detected"] for c in sel]) >= 0.5)
            c_ = _floor(cells, arc, s, masses,
                        lambda sel: np.median([c["log_mass_error"] for c in sel]) < char_thresh)
            if d:
                det_x.append(s); det_y.append(d)
            if c_:
                char_x.append(s); char_y.append(c_)
        if det_x:
            ax.plot(det_x, det_y, "o-", color=col, lw=2, label=f"{arc:g}p detect")
        if char_x:
            ax.plot(char_x, char_y, "s--", color=col, lw=1.5, alpha=0.8,
                    label=f"{arc:g}p charac.")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("observation noise sigma")
    ax.set_ylabel("min hidden mass")
    ax.set_title(f"Sensitivity floors (solid=detect, dashed=recover to <{char_thresh} dex)")
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8, ncol=len(arcs))
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight"); plt.close(fig)
    return path


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path


### `src/perturber/runner.py`

In [ ]:
"""Experiment orchestration: generate -> fit (perturber + null) -> evaluate -> plots.

Shared by the CLI scripts and the built Kaggle notebook.
"""
import hashlib
import json
import os
from dataclasses import asdict, replace

import numpy as np



def run_single_experiment(sys_cfg, fit_cfg, outdir, make_plot=True, verbose=True):
    """One full experiment. Returns the combined metrics dict."""
    ensure_dir(outdir)
    ds = generate(sys_cfg)
    if verbose:
        print(f"[run] arc {sys_cfg.n_periods:g} periods, {len(ds.t)} obs, "
              f"sigma {sys_cfg.sigma:.0e}, hidden mass {sys_cfg.hidden_mass:.0e}")

    results = {}
    for kind in ("perturber", "null"):
        # The null model has no randomized parameters — every restart is
        # identical, so one is enough. Only the perturber benefits from
        # multi-start over the nonconvex hidden-orbit landscape.
        cfg_k = fit_cfg if kind == "perturber" else replace(fit_cfg, n_restarts=1)
        if verbose:
            print(f"[fit] {kind} model, {cfg_k.n_restarts} restarts")
        model = make_model(kind, ds, cfg_k)
        res = fit(model, ds, cfg_k, verbose=verbose)
        metrics, traj = evaluate(model, res, ds, cfg_k)
        results[kind] = (metrics, traj)

    metrics_pert, traj_pert = results["perturber"]
    metrics_null, traj_null = results["null"]
    cmp = compare(metrics_pert, metrics_null)

    out = {"perturber": metrics_pert, "null": metrics_null, "comparison": cmp,
           "system": asdict(sys_cfg)}
    with open(os.path.join(outdir, "metrics.json"), "w") as f:
        json.dump(out, f, indent=2)
    dump_configs(sys_cfg, fit_cfg, os.path.join(outdir, "config.json"))
    if make_plot:
        p = plot_experiment(ds, traj_pert, traj_null, metrics_pert, metrics_null,
                            cmp, os.path.join(outdir, "experiment.png"))
        if verbose:
            print(f"[run] figure -> {p}")

    if verbose:
        print(f"[run] mass true {metrics_pert['mass_true']:.2e} "
              f"recovered {metrics_pert['mass_recovered']:.2e} "
              f"(log err {metrics_pert['log_mass_error']:.3f}) | "
              f"2dlogL {cmp['loglike_ratio_stat']:.1f} "
              f"{'DETECTED' if cmp['detected'] else 'not detected'}")
    return out


def threshold_grid(preset):
    """(masses, sigmas, arcs, seeds) for the sweep.

    'kaggle' is the full science grid (5x3x3x3 = 135 cells). 'core' drops the
    unrealistically-low-noise row and one seed for a ~feasible 4x2x3x2 = 48-cell
    map. 'smoke' is a 2-cell sanity grid.
    """
    if preset == "smoke":
        return [1e-2, 3e-3], [1e-4], [4.0], [0]
    if preset == "core":
        return ([1e-2, 1e-3, 3e-4, 1e-4], [1e-4, 1e-3], [8.0, 16.0, 40.0], [0, 1])
    # full grid — realistic noise only. sigma=1e-5 (10 ppm astrometry) is excluded:
    # it is physically unrealistic AND a first-order fit can't reach its noise floor
    # in a practical step budget, producing spurious non-detections rather than a
    # physical boundary (confirmed by a fidelity sweep on m=3e-3/arc=8).
    # Mass range extends below the visible planets (1e-4, 3e-5) down to 3e-6 to
    # locate the actual detection floor (detection is robust down to ~1e-4).
    return ([1e-2, 3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6],
            [1e-4, 3e-4, 1e-3],
            [8.0, 16.0, 40.0],
            [0, 1, 2])


def sweep_substeps(sigma):
    """RK4 substeps chosen so integration truncation error stays well below the
    noise floor (RK4 error ~ substeps^-4; empirically ~6e-5 at substeps=2)."""
    if sigma <= 1e-5:
        return 6
    if sigma <= 3e-4:      # covers 1e-4 and 3e-4: integration error ~2.6e-6 << noise
        return 4
    return 2               # sigma >= 1e-3


def sweep_fit_config(base_fit, sigma, seed):
    """Reduced-fidelity fit for sweep cells: CPU (3x faster than the T400 GPU on
    this Python-loop-bound workload), 8 restarts, trimmed curriculum, no MS,
    integration fidelity scaled to the noise."""
    return replace(base_fit, seed=seed, device="cpu", n_restarts=12, ms_steps=0,
                   substeps=sweep_substeps(sigma),
                   curriculum=((0.25, 120), (0.5, 120), (1.0, 250)))


def _collect_and_plot(outdir):
    """Render the heatmap from every cell JSON present in outdir (robust to
    sharding and cross-session resume — always plots the full merged set)."""
    cells = []
    for fn in sorted(os.listdir(outdir)):
        if fn.endswith(".json") and fn != "threshold_meta.json":
            with open(os.path.join(outdir, fn)) as f:
                obj = json.load(f)
            if "mass" in obj:  # a cell record, not some other artifact
                cells.append(obj)
    if cells:
        plot_threshold(cells, os.path.join(outdir, "threshold.png"))
    return cells


def sweep_cell_specs(preset, outdir, shard=0, n_shards=1, grid=None):
    """Build the (picklable) work list for the sweep. Each spec carries the
    fully-resolved sys/fit configs and the cell's output path — enough for a
    worker process to run it with no shared state."""
    grid = grid or ("smoke" if preset == "smoke" else preset)
    base_sys, base_fit = get_preset("smoke" if preset == "smoke" else "kaggle")
    masses, sigmas, arcs, seeds = threshold_grid(grid)
    todo = [(m, s, a, sd) for m in masses for s in sigmas for a in arcs for sd in seeds]

    specs = []
    for i, (m, s, a, sd) in enumerate(todo):
        if i % n_shards != shard:
            continue
        sys_cfg = replace(base_sys, hidden_mass=m, sigma=s, n_periods=a, seed=sd)
        fit_cfg = replace(base_fit, seed=sd) if preset == "smoke" \
            else sweep_fit_config(base_fit, s, sd)
        key = hashlib.md5(json.dumps(
            {**asdict(sys_cfg), "fit": asdict(fit_cfg)}, sort_keys=True
        ).encode()).hexdigest()[:12]
        specs.append({"sys": sys_cfg, "fit": fit_cfg, "key": key,
                      "cell_path": os.path.join(outdir, f"{key}.json"),
                      "run_dir": os.path.join(outdir, f"run_{key}")})
    return specs, len(todo)


def execute_cell(spec, verbose=False, threads=None):
    """Run one sweep cell (skips if its JSON already exists). Module-level and
    self-contained so it can be dispatched to a worker process. Returns the
    cell dict."""
    if os.path.exists(spec["cell_path"]):
        with open(spec["cell_path"]) as f:
            return json.load(f)
    if threads is not None:
        import torch
        torch.set_num_threads(threads)   # avoid oversubscription under a process pool
    sys_cfg, fit_cfg = spec["sys"], spec["fit"]
    print(f"[cell] m={sys_cfg.hidden_mass:.0e} sigma={sys_cfg.sigma:.0e} "
          f"arc={sys_cfg.n_periods:g} seed={sys_cfg.seed} "
          f"substeps={fit_cfg.substeps}", flush=True)
    r = run_single_experiment(sys_cfg, fit_cfg, spec["run_dir"],
                              make_plot=False, verbose=verbose)
    cell = {"mass": sys_cfg.hidden_mass, "sigma": sys_cfg.sigma,
            "n_periods": sys_cfg.n_periods, "seed": sys_cfg.seed,
            "log_mass_error": r["perturber"]["log_mass_error"],
            "detected": r["comparison"]["detected"],
            "loglike_ratio_stat": r["comparison"]["loglike_ratio_stat"],
            "delta_rmse_test": r["comparison"]["delta_rmse_test"]}
    with open(spec["cell_path"], "w") as f:
        json.dump(cell, f, indent=2)
    return cell


def run_threshold_study(preset, outdir, verbose=False, shard=0, n_shards=1,
                        grid=None):
    """Resumable, sharded detection-threshold sweep (sequential).

    One JSON per grid cell; existing cells are skipped, so the sweep resumes
    after any interruption and can be split across sessions/machines via
    (shard, n_shards): each shard runs cell i where i % n_shards == shard.
    Copy prior sessions' cell JSONs into outdir before starting to resume them.
    For local multi-core runs use scripts/run_sweep_parallel.py instead.
    """
    ensure_dir(outdir)
    specs, n_total = sweep_cell_specs(preset, outdir, shard, n_shards, grid)
    print(f"[sweep] {n_total} cells total, shard {shard}/{n_shards} "
          f"handles {len(specs)}", flush=True)
    for j, spec in enumerate(specs):
        print(f"[sweep {j + 1}/{len(specs)}]", flush=True)
        execute_cell(spec, verbose=verbose)
    cells = _collect_and_plot(outdir)
    print(f"[sweep] {len(cells)} cells present in {outdir} -> threshold.png")
    return cells


### Run

In [ ]:
# ── Driver ────────────────────────────────────────────────────────────────
import os, glob, shutil

SMOKE = os.environ.get("PERTURBER_SMOKE") == "1"   # fast validation run

# --- What to run -----------------------------------------------------------
RUN_SWEEP  = False        # False: one demo experiment | True: detection-threshold sweep
GRID       = "core"       # "core" (48 cells, feasible) | "kaggle" (135, full science)
SHARD      = 0            # this session's shard index ...
N_SHARDS   = 1            # ... of N parallel sessions (each does cells i%N==SHARD)
# ---------------------------------------------------------------------------

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "results_nb"

if not RUN_SWEEP:
    preset = "smoke" if SMOKE else "kaggle"
    sys_cfg, fit_cfg = get_preset(preset)
    result = run_single_experiment(sys_cfg, fit_cfg, os.path.join(OUT_DIR, "experiment"))
else:
    # The sweep is CPU-bound (tiny per-step tensors -> kernel-launch overhead
    # makes a GPU slower here). Turn OFF the Kaggle accelerator for sweep runs.
    sweep_dir = os.path.join(OUT_DIR, "threshold")
    os.makedirs(sweep_dir, exist_ok=True)
    # Resume: copy any prior cell JSONs from attached input datasets so already-
    # computed cells are skipped. Attach a previous run's output as a dataset.
    for prior in glob.glob("/kaggle/input/*/threshold/*.json"):
        dst = os.path.join(sweep_dir, os.path.basename(prior))
        if not os.path.exists(dst):
            shutil.copy(prior, dst)
    run_threshold_study("smoke" if SMOKE else "kaggle", sweep_dir,
                        grid="smoke" if SMOKE else GRID,
                        shard=SHARD, n_shards=N_SHARDS)
